In [8]:
# 絵を空中に描くプログラム

import cv2
import mediapipe as mp
import math
import numpy as np
import time

# ========== 設定 ==========
# ピンチと判定する距離の閾値
PINCH_THRESHOLD = 0.05 
# ==========================

def map_coordinate(val, min_val, max_val, out_max):
    """MediaPipeの座標を、指定した範囲で切り取って画面サイズに変換する"""
    # ゼロ割り算エラーを防ぐ
    if max_val - min_val == 0:
        return 0
    norm_val = (val - min_val) / (max_val - min_val)
    norm_val = max(0.0, min(1.0, norm_val))
    return int(norm_val * out_max)

def main():
    # MediaPipeの準備
    mp_hands = mp.solutions.hands
    # 管理者画面用にMediaPipe標準の描画ツールを用意
    mp_drawing = mp.solutions.drawing_utils 
    
    hands = mp_hands.Hands(
        min_detection_confidence=0.7,
        min_tracking_confidence=0.7,
        max_num_hands=1
    )

    cap = cv2.VideoCapture(0) # 外付けカメラの場合はここを変更
    
    ret = False
    for i in range(10): # 最大10回（約1秒）リトライする
        ret, frame = cap.read()
        if ret:
            break # 成功したらループを抜ける
        time.sleep(0.1) # 0.1秒待って再挑戦
        
    if not ret:
        print("カメラが見つかりません。")
        return
        
    h, w, c = frame.shape
    canvas = np.zeros((h, w, 3), dtype=np.uint8)

    # === キャリブレーション用の変数 ===
    is_calibrating = True  # 起動時は必ずキャリブレーションからスタート
    calib_step = 0         # 現在何個目の的か (0〜3)
    calib_points = []      # 取得した4点の座標を保存するリスト
    was_pinching = False   # 連続ピンチを防ぐためのフラグ
    last_calib_time = 0.0  # 最後に的をピンチした時間を記録（誤作動防止用）

    # デフォルトのマッピング範囲（キャリブレーション完了後に上書きされます）
    map_min_x, map_max_x = 0.2, 0.8
    map_min_y, map_max_y = 0.2, 0.8

    # 4つの的の画面上の位置（左上、右上、左下、右下）
    targets = [
        (int(w * 0.1), int(h * 0.1)),
        (int(w * 0.9), int(h * 0.1)),
        (int(w * 0.1), int(h * 0.9)),
        (int(w * 0.9), int(h * 0.9))
    ]

    prev_x, prev_y = 0, 0

    print("=== 操作方法 ===")
    print("・親指と人差し指を近づけると線が描けます（お絵描きモード時）")
    print("・'R' キー：キャリブレーションを開始 / やり直し")
    print("・'C' キー：キャンバスを消去（クリア）")
    print("・'F' キー：プログラム終了")
    print("================")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.flip(frame, 1)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb_frame)

        # プロジェクター表示用（真っ黒キャンバス）
        display_image = canvas.copy()
        
        # PCモニター確認用（カメラ映像）
        admin_frame = frame.copy()

        # キャリブレーション中はマッピング範囲を「カメラ全域(0.0~1.0)」にする
        if is_calibrating:
            current_min_x, current_max_x = 0.0, 1.0
            current_min_y, current_max_y = 0.0, 1.0
            
            target_x, target_y = targets[calib_step]
            # 的（二重丸）をプロジェクター用に描画
            cv2.circle(display_image, (target_x, target_y), 20, (255, 255, 0), 2)
            cv2.circle(display_image, (target_x, target_y), 5, (255, 255, 0), cv2.FILLED)
            
            # 管理者画面にも状態を表示
            cv2.putText(admin_frame, f"CALIBRATING... {calib_step}/4", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
        else:
            # お絵描き中は、キャリブレーションで計算した専用の範囲を使う
            current_min_x, current_max_x = map_min_x, map_max_x
            current_min_y, current_max_y = map_min_y, map_max_y
            
            # デバッグ用の仮想キャンバス枠（プロジェクター用は暗めのグレー）
            cv2.rectangle(display_image, 
                          (int((0.0 - current_min_x)/(current_max_x - current_min_x)*w), int((0.0 - current_min_y)/(current_max_y - current_min_y)*h)), 
                          (w, h), (50, 50, 50), 2)
            
            # 管理者画面に「ユーザー専用の仮想キャンバス領域」を青枠で表示（画角確認用）
            cv2.rectangle(admin_frame, 
                          (int(current_min_x * w), int(current_min_y * h)), 
                          (int(current_max_x * w), int(current_max_y * h)), 
                          (255, 0, 0), 2)
            cv2.putText(admin_frame, "Drawing Mode", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                
                # 管理者画面に標準の骨格を描画（手が正しく認識できているかの確認用）
                mp_drawing.draw_landmarks(admin_frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
                
                # 21個の関節座標の計算は裏で行う（ポインターの位置を出すため）
                mapped_landmarks = []
                for lm in hand_landmarks.landmark:
                    cx = map_coordinate(lm.x, current_min_x, current_max_x, w)
                    cy = map_coordinate(lm.y, current_min_y, current_max_y, h)
                    mapped_landmarks.append((cx, cy))
                
                # ==== ハンドメッシュの描画（お絵描き中のみプロジェクターに表示） ====
                if not is_calibrating:
                    for connection in mp_hands.HAND_CONNECTIONS:
                        p1 = mapped_landmarks[connection[0]]
                        p2 = mapped_landmarks[connection[1]]
                        cv2.line(display_image, p1, p2, (0, 255, 0), 5)
                    
                    for p in mapped_landmarks:
                        cv2.circle(display_image, p, 7, (0, 0, 255), cv2.FILLED)
                # ====================================================

                thumb = hand_landmarks.landmark[mp_hands.HandLandmark.THUMB_TIP]
                index = hand_landmarks.landmark[mp_hands.HandLandmark.INDEX_FINGER_TIP]

                # ペン先の座標を取得
                pointer_x, pointer_y = mapped_landmarks[mp_hands.HandLandmark.INDEX_FINGER_TIP]
                distance = math.sqrt((thumb.x - index.x)**2 + (thumb.y - index.y)**2)
                is_pinching = distance < PINCH_THRESHOLD

                if is_pinching:
                    if is_calibrating:
                        current_time = time.time()
                        # 指を一度離したか、かつ前回から1秒以上経過しているかを確認
                        if not was_pinching and (current_time - last_calib_time > 1.0):
                            # 生の座標(0.0~1.0)を記録
                            calib_points.append((index.x, index.y))
                            calib_step += 1
                            last_calib_time = current_time # 最後にピンチした時間を更新
                            
                            # 4点すべて完了したら範囲を再計算
                            if calib_step >= 4:
                                map_min_x = (calib_points[0][0] + calib_points[2][0]) / 2
                                map_max_x = (calib_points[1][0] + calib_points[3][0]) / 2
                                map_min_y = (calib_points[0][1] + calib_points[1][1]) / 2
                                map_max_y = (calib_points[2][1] + calib_points[3][1]) / 2
                                
                                # 異常値防止
                                if map_max_x <= map_min_x: map_max_x = map_min_x + 0.1
                                if map_max_y <= map_min_y: map_max_y = map_min_y + 0.1
                                
                                is_calibrating = False
                                canvas = np.zeros((h, w, 3), dtype=np.uint8) # キャンバスを綺麗にして開始
                    else:
                        # お絵描きモード：ペンON（赤ポインターを表示して線を引く）
                        cv2.circle(display_image, (pointer_x, pointer_y), 8, (0, 0, 255), cv2.FILLED)
                        if prev_x != 0 and prev_y != 0:
                            cv2.line(canvas, (prev_x, prev_y), (pointer_x, pointer_y), (0, 255, 255), 10)
                        prev_x, prev_y = pointer_x, pointer_y
                
                else:
                    if not is_calibrating:
                        # お絵描きモード：ペンOFF（白ポインターを表示）
                        cv2.circle(display_image, (pointer_x, pointer_y), 12, (255, 255, 255), 2)
                    prev_x, prev_y = 0, 0

                # ピンチ状態を保存（次のフレームで比較するため）
                was_pinching = is_pinching

        # ===== ウィンドウの表示 =====
        # 1. 体験者用（プロジェクターへ持っていく画面）
        cv2.imshow('AI Drawing MVP (Projector)', display_image)
        # 2. 管理者用（Macの画面に残しておく画面）
        cv2.imshow('Admin Camera View (MacBook)', admin_frame)

        # キー操作の受付
        key = cv2.waitKey(1) & 0xFF
        if key == ord('c') or key == ord('C'):
            canvas = np.zeros((h, w, 3), dtype=np.uint8)
        elif key == ord('r') or key == ord('R'):
            # キャリブレーションをリセットして再開
            is_calibrating = True
            calib_step = 0
            calib_points = []
            last_calib_time = time.time() # リセット直後に誤作動しないよう時間を入れる
            canvas = np.zeros((h, w, 3), dtype=np.uint8)
        elif key == ord('f') or key == ord('F'):
            break

    cap.release()
    cv2.destroyAllWindows()

    cv2.waitKey(1)
    cv2.waitKey(1)
    cv2.waitKey(1)
    cv2.waitKey(1)

if __name__ == "__main__":
    main()

I0000 00:00:1786081864.165602   57512 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2 Pro
W0000 00:00:1786081864.173489   61447 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1786081864.178894   61450 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


=== 操作方法 ===
・親指と人差し指を近づけると線が描けます（お絵描きモード時）
・'R' キー：キャリブレーションを開始 / やり直し
・'C' キー：キャンバスを消去（クリア）
・'F' キー：プログラム終了
